# 15 · Ground truth and model upgrades (LABELS_PLAN.md, MODEL_UPGRADE_PLAN.md)
Every adoption rule is written in the module docstring before the run.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Multi-source label set (IMD + SANDRP + Sphere India SitReps + EM-DAT + curated news)

In [2]:
print(subprocess.run([sys.executable, 'scripts/build_labels.py'], capture_output=True, text=True).stdout)
pd.read_csv('data/external/labels/labels_2022_2025.csv').head(20)

{
  "events": 96,
  "district_days": 130,
  "by_year": {
    "2022": 28,
    "2023": 40,
    "2024": 11,
    "2025": 17
  },
  "by_confidence": {
    "high": 88,
    "low": 6,
    "medium": 2
  },
  "by_source_family": {
    "curated": 14,
    "emdat": 25,
    "imd": 29,
    "sandrp": 52,
    "sphere": 44
  },
  "corroborated_events": 22,
  "excluded": 1,
  "unresolved": 0
}



,event_id,date,end_date,district,place,type,hour_local,deaths,severity,date_confidence,sources,source_count,n_reports,source_ref
0,EV-20220507-NAIN,2022-05-07,2022-05-07,Nainital,NaN,cloudburst,NaN,NaN,high,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
1,EV-20220510-BAGE,2022-05-10,2022-05-10,Bageshwar,NaN,flash_flood,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
2,EV-20220627-NAIN,2022-06-27,2022-06-27,Nainital,NaN,cloudburst,NaN,NaN,low,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
3,EV-20220705-UTTA,2022-07-05,2022-07-05,Uttarkashi,NaN,cloudburst,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
4,EV-20220708-BAGE,2022-07-08,2022-07-08,Bageshwar,NaN,cloudburst,NaN,NaN,high,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
5,EV-20220708-PITH,2022-07-08,2022-07-08,Pithoragarh,NaN,cloudburst,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
6,EV-20220709-NAIN,2022-07-09,2022-07-09,Nainital,NaN,flash_flood,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
7,EV-20220714-DEHR,2022-07-14,2022-07-14,Dehradun,NaN,flash_flood,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html
8,EV-20220719-RUDR,2022-07-19,2022-07-19,Rudraprayag,NaN,heavy_rain_flood,NaN,2.0,high,high,imd,1,1,UEI-IMD-FL-2022-1098
9,EV-20220720-CHAM,2022-07-20,2022-07-20,Chamoli,NaN,cloudburst,NaN,NaN,moderate,high,sandrp,1,1,sandrp/uttarakhand_cloudbursts_2022.html


## 2. Audit: IMD vs SANDRP on the overlap period

In [3]:
from src.label_audit import run as audit_run
audit_run();

                   imd_days  sandrp_days  matched_imd_days  sandrp_only_days
district                                                                    
Bageshwar                 0            4                 0                 4
Chamoli                   2            9                 1                 8
Dehradun                  5            5                 1                 4
Nainital                  0            5                 0                 5
Pauri Garhwal             3            8                 2                 6
Pithoragarh               1            6                 0                 6
Rudraprayag               9            5                 2                 3
Tehri Garhwal             4            3                 2                 1
Udham Singh Nagar         1            0                 0                 0
Uttarkashi                3            5                 0                 5
{
  "overlap_period": [
    "2022-05-01..2022-10-31",
    "2023-06-01..2023-

## 3. Label v1 vs v2 (2×2 train-label × truth grid; slow, so results are read from the last run of src.label_experiment)

In [4]:
e = json.loads(Path('results/labels/label_experiment.json').read_text())
display(pd.DataFrame(e['grid']).query("model == 'XGBoost'"))
e['decision']

,model,trained_on,scored_on,threshold,roc_auc,pr_auc,precision,recall,f1,test_base_rate,dd_POD,dd_FAR,dd_CSI,dd_HSS,dd_CSI_ci95,dd_tp_fp_fn
0,XGBoost,v1,v1,0.25,0.7076,0.1501,0.2223,0.5736,0.3204,0.0577,0.474,0.820,0.150,0.179,"[0.085, 0.223]",9/41/10
1,XGBoost,v1,v2,0.25,0.6470,0.2059,0.3428,0.4694,0.3963,0.1088,0.400,0.720,0.197,0.212,"[0.124, 0.292]",14/36/21
2,XGBoost,v1,v2_masked,0.25,0.6421,0.2157,0.3595,0.4694,0.4072,0.1191,0.400,0.696,0.209,0.226,"[0.129, 0.305]",14/32/21
3,XGBoost,v2,v1,0.65,0.6858,0.1126,0.1220,0.2444,0.1627,0.0577,0.368,0.857,0.115,0.118,"[0.051, 0.183]",7/42/12
4,XGBoost,v2,v2,0.65,0.6371,0.1844,0.2426,0.2579,0.2500,0.1088,0.343,0.755,0.167,0.162,"[0.097, 0.253]",12/37/23
5,XGBoost,v2,v2_masked,0.65,0.6338,0.1982,0.2684,0.2579,0.2631,0.1191,0.343,0.733,0.176,0.174,"[0.105, 0.265]",12/33/23
6,XGBoost,v2_masked,v1,0.70,0.6865,0.1135,0.1220,0.2444,0.1627,0.0577,0.368,0.857,0.115,0.118,"[0.051, 0.183]",7/42/12
7,XGBoost,v2_masked,v2,0.70,0.6363,0.1826,0.2426,0.2579,0.2500,0.1088,0.343,0.755,0.167,0.162,"[0.097, 0.253]",12/37/23
8,XGBoost,v2_masked,v2_masked,0.70,0.6332,0.1968,0.2684,0.2579,0.2631,0.1191,0.343,0.733,0.176,0.174,"[0.105, 0.265]",12/33/23


{'rule': 'adopt v2 if, for XGBoost, the v2-trained model is at least as good as v1-trained on BOTH truths (PR-AUC and district-day CSI) and event-CV PR-AUC improves',
 'v2_on_v1_truth_ok': False,
 'v2_on_v2_truth_ok': False,
 'event_cv_ok': True,
 'adopt_v2': False}

## 4. District-day model (CWC + USDMA hydromet + ERA5)

In [5]:
from src.district_day import run as dd_run
dd = dd_run()
dd['decision']

{'train': (6695, 18), 'validation': (403, 15), 'test': (325, 22)}


{
 "model": "logistic",
 "validation_pr_auc": {
  "logistic": 0.1968,
  "xgb_monotone": 0.0852
 },
 "threshold": 0.9744,
 "counts": {
  "train": {
   "district_days": 6695,
   "positives": 18
  },
  "validation": {
   "district_days": 403,
   "positives": 15
  },
  "test": {
   "district_days": 325,
   "positives": 22
  }
 },
 "test_all": {
  "roc_auc": 0.782,
  "pr_auc": 0.24,
  "base_rate": 0.068,
  "brier": 0.2505,
  "tp": 3,
  "fp": 7,
  "fn": 19,
  "tn": 296,
  "POD": 0.136,
  "FAR": 0.7,
  "POFD": 0.023,
  "CSI": 0.103,
  "ETS": 0.082,
  "HSS": 0.152,
  "bias": 0.455,
  "CSI_ci95": [
   0.0,
   0.232
  ]
 },
 "test_common_with_hybrid": {
  "district_days": 276,
  "district_day_model": {
   "tp": 3,
   "fp": 7,
   "fn": 16,
   "tn": 250,
   "POD": 0.158,
   "FAR": 0.7,
   "POFD": 0.027,
   "CSI": 0.115,
   "ETS": 0.091,
   "HSS": 0.167,
   "bias": 0.526,
   "CSI_ci95": [
    0.0,
    0.259
   ]
  },
  "hybrid": {
   "tp": 12,
   "fp": 52,
   "fn": 7,
   "tn": 205,
   "POD": 0.632,

{'csi_ok': False, 'external_pod_ok': False, 'burden_ok': True, 'adopt': False}

## 5. E1 ensemble · E2 event-fitted rain thresholds · E3 pooled district-day + ranking

In [6]:
from src.ensemble import run as e1
from src.rain_thresholds import run as e2
from src.district_pooled import run as e3
r1, r2, r3 = e1(), e2(), e3()

xgb {'v1': {'roc_auc': 0.7076, 'pr_auc': 0.1501, 'f1': 0.3204, 'dd_POD': 0.474, 'dd_FAR': 0.82, 'dd_CSI': 0.15}, 'v2': {'roc_auc': 0.647, 'pr_auc': 0.2059, 'f1': 0.3963, 'dd_POD': 0.4, 'dd_FAR': 0.72, 'dd_CSI': 0.197}}
ens {'v1': {'roc_auc': 0.7842, 'pr_auc': 0.1511, 'f1': 0.3091, 'dd_POD': 0.474, 'dd_FAR': 0.827, 'dd_CSI': 0.145}, 'v2': {'roc_auc': 0.7355, 'pr_auc': 0.2323, 'f1': 0.3829, 'dd_POD': 0.4, 'dd_FAR': 0.731, 'dd_CSI': 0.192}}
rf {'v1': {'roc_auc': 0.7881, 'pr_auc': 0.1523, 'f1': 0.2357, 'dd_POD': 0.474, 'dd_FAR': 0.82, 'dd_CSI': 0.15}, 'v2': {'roc_auc': 0.7285, 'pr_auc': 0.2329, 'f1': 0.3068, 'dd_POD': 0.4, 'dd_FAR': 0.72, 'dd_CSI': 0.197}}
{'xgb': {'pooled_roc_auc': 0.7006, 'pooled_pr_auc': 0.1879}, 'ens': {'pooled_roc_auc': 0.7321, 'pooled_pr_auc': 0.2046}, 'folds': 6} {'event_cv_ok': True, 'test_v1_ok': False, 'test_v2_ok': False, 'adopt': False}


{
 "thresholds": {
  "n_fit_events": 46,
  "pooled": {
   "r24": {
    "ALERT": 12.0,
    "SEVERE": 47.5,
    "EXTREME": 108.5
   },
   "r72": {
    "ALERT": 23.0,
    "SEVERE": 91.5,
    "EXTREME": 183.0
   }
  },
  "plains": {
   "r24": {
    "ALERT": 12.0,
    "SEVERE": 47.5,
    "EXTREME": 108.5
   },
   "r72": {
    "ALERT": 23.0,
    "SEVERE": 91.5,
    "EXTREME": 183.0
   },
   "n": 3
  },
  "hills": {
   "r24": {
    "ALERT": 12.0,
    "SEVERE": 42.4,
    "EXTREME": 108.3
   },
   "r72": {
    "ALERT": 22.4,
    "SEVERE": 87.0,
    "EXTREME": 177.8
   },
   "n": 43
  }
 },
 "current": {
  "CSI": 0.169,
  "POD": 0.632,
  "FAR": 0.812,
  "pod_2025": "2/10",
  "alert_dd": {
   "2024": 370,
   "2025": 171
  }
 },
 "fitted": {
  "CSI": 0.088,
  "POD": 0.842,
  "FAR": 0.911,
  "pod_2025": "5/10",
  "alert_dd": {
   "2024": 808,
   "2025": 406
  }
 },
 "decision": {
  "csi_ok": false,
  "pod_2025_ok": true,
  "burden_ok": false,
  "adopt": false
 }
}


{
 "threshold": 0.964,
 "test": {
  "roc_auc": 0.753,
  "pr_auc": 0.155,
  "base_rate": 0.068,
  "tp": 4,
  "fp": 20,
  "fn": 18,
  "tn": 283,
  "POD": 0.182,
  "FAR": 0.833,
  "POFD": 0.066,
  "CSI": 0.095,
  "ETS": 0.059,
  "HSS": 0.111,
  "bias": 1.091
 },
 "ranking_recall_at_3": {
  "test_model": 0.455,
  "test_prior": 0.409,
  "external_model": 0.345,
  "external_prior": 0.207,
  "chance": 0.231
 },
 "decision": {
  "csi_ok": false,
  "external_pod_ok": false,
  "burden_ok": true,
  "adopt_alert_layer": false,
  "offer_ranking_view": true
 }
}
test common: {'pooled': {'POD': 0.211, 'FAR': 0.826, 'CSI': 0.105}, 'hybrid': {'POD': 0.632, 'FAR': 0.812, 'CSI': 0.169}} | external: {'external_pooled': (3, 29, [16, 98, 58, 83]), 'external_hybrid': (7, 21, [81, 370, 171, 318])}


In [7]:
print(subprocess.run([sys.executable, 'scripts/model_upgrade_report.py'], capture_output=True, text=True).stdout[:200])
display(Markdown(Path('results/model_upgrade.md').read_text(encoding='utf-8')))

# PRAHAR: model upgrade experiments (MODEL_UPGRADE_PLAN.md)

Every adoption rule is written in the module docstring before the first run; the same splits and verification code are used throughout.

##


# PRAHAR: model upgrade experiments (MODEL_UPGRADE_PLAN.md)

Every adoption rule is written in the module docstring before the first run; the same splits and verification code are used throughout.

## E1: equal-weight ensemble (LR + RF + XGBoost, percentile-averaged)

| Model | ROC-AUC v1 | PR-AUC v1 | Day CSI v1 | PR-AUC v2 | Day CSI v2 |
|---|---|---|---|---|---|
| xgb | 0.7076 | 0.1501 | 0.15 | 0.2059 | 0.197 |
| rf | 0.7881 | 0.1523 | 0.15 | 0.2329 | 0.197 |
| ens | 0.7842 | 0.1511 | 0.145 | 0.2323 | 0.192 |

Event-CV pooled PR-AUC: XGBoost 0.1879, ensemble 0.2046 (ROC 0.7006 → 0.7321).
Decision: **adopt = False** {'event_cv_ok': True, 'test_v1_ok': False, 'test_v2_ok': False, 'adopt': False}. Ranking improves; district-day CSI does not.

## E2: rainfall thresholds fitted on documented events

Fit on 46 events (2022 – Jul 2023 and 2024). Hills, 24 h: ALERT 12.0 · SEVERE 42.4 · EXTREME 108.3 mm; 72 h: 22.4 · 87.0 · 177.8 mm (plains: too few events, so pooled values are used).
**Finding:** half of the documented floods had < 50 mm/24 h at the wettest gauge in their district, so most were local cloudbursts the gauge network did not record. Rain at the gauge cannot separate flood days from wet days.

| Rule | Test day POD | FAR | CSI | 2025 events POD | ALERT district-days 2024 / 2025 |
|---|---|---|---|---|---|
| current | 0.632 | 0.812 | 0.169 | 2/10 | 370 / 171 |
| fitted | 0.842 | 0.911 | 0.088 | 5/10 | 808 / 406 |

Decision: **adopt = False** {'csi_ok': False, 'pod_2025_ok': True, 'burden_ok': False, 'adopt': False}.

## E3: pooled district-day model + ranking view

Test ROC-AUC 0.753, PR-AUC 0.155 (base rate 0.068). As an alert layer at frequency bias 2: CSI 0.105 vs hybrid 0.169; external 3/29 vs hybrid 7/21.
Ranking, recall@3 (flooded district among the day's top 3): test 0.455 vs district prior 0.409; external events 0.345 vs prior 0.207 (chance 0.231).
Decision: alert layer adopt = **False**; ranking view offered = **True** (Overview → 'Districts to watch').

## Conclusion

No alert-layer change beats the deployed hybrid under its pre-set rule. The limit is the labels (about 50 IMD flood district-days) and the gauge network's blindness to local cloudbursts, not the model class. Two things do improve and are kept as reported results: ranking (ensemble ROC-AUC 0.78; district ranking recall@3 0.34 vs 0.21 prior). Next steps are data: a complete 2024–25 event list (USDMA SEOC daily reports via RTI) and radar or satellite rain (IMD Doppler or GPM IMERG) that can see cloudbursts between gauges.